In [1]:
import os
api = os.environ["API"]

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from typing import Annotated
from langchain_core.tools import InjectedToolArg

In [26]:
@tool
def convert_currency(base_currency: str, target_currency: str) -> float:
    """This function fetches the exchange rate between two currencies. it takes two parameters: base_currency and target_currency. It returns the exchange rate as a float."""
    url = f"https://v6.exchangerate-api.com/v6/{api}/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()

@tool
def convert(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """Given the the conversion rate this function is to convert the currency value based on the conversion rate.
    It takes two parameters: base_currency_value and conversion_rate.
    It returns the converted amount."""
    return base_currency_value * conversion_rate

# @tool 
# def convert(base_currency_value: float, conversion_rate: float) -> float:
#     """This function is to convert the currency value based on the conversion rate.
#     It takes two parameters: base_currency_value and conversion_rate.
#     It returns the converted amount."""
#     return base_currency_value * conversion_rate



In [27]:
# testing API
url = f"https://v6.exchangerate-api.com/v6/{api}/pair/USD/INR"
response = requests.get(url)
response.json()

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791849601,
 'time_last_update_utc': 'Tue, 13 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1791936001,
 'time_next_update_utc': 'Wed, 14 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.9069}

In [28]:
convert_currency.invoke({'base_currency': 'USD', 'target_currency': 'INR'})


{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791849601,
 'time_last_update_utc': 'Tue, 13 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1791936001,
 'time_next_update_utc': 'Wed, 14 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.9069}

In [29]:
convert.invoke({'base_currency_value': 100, 'conversion_rate': 82.5})

8250.0

In [30]:
llm = ChatOllama(model="qwen2.5:3b", temperature=0)

In [31]:
llm_tools = llm.bind_tools([convert_currency, convert])

In [32]:
messages = [HumanMessage("What is the conversion rate of USD to INR? Can you convert 100 USD to INR?") ]

In [33]:
messages 

[HumanMessage(content='What is the conversion rate of USD to INR? Can you convert 100 USD to INR?', additional_kwargs={}, response_metadata={})]

In [34]:
llm_tools.invoke(messages)

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-13T03:48:05.016307Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7902133166, 'load_duration': 47770333, 'prompt_eval_count': 277, 'prompt_eval_duration': 1129605000, 'eval_count': 96, 'eval_duration': 6695665000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a1335f-c60a-7002-b585-2004c4535204-0', tool_calls=[{'name': 'convert_currency', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '4085417b-0650-4a87-bd2a-14e0bb115e42', 'type': 'tool_call'}, {'name': 'convert', 'args': {'conversion_rate': 82.56, 'base_currency_value': 100}, 'id': 'decfff47-bb16-42f3-b0b0-24822299a0de', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 277, 'output_tokens': 96, 'total_tokens': 373})

In [35]:
llm_tools.invoke(messages).tool_calls

[{'name': 'convert_currency',
  'args': {'target_currency': 'INR', 'base_currency': 'USD'},
  'id': 'd0ef6d7a-49f2-4c61-a851-ca6c640b66bc',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 100, 'conversion_rate': 82.56},
  'id': 'fa503531-7f07-477b-8a57-cbadb649eca0',
  'type': 'tool_call'}]

In [37]:
ai_messages = llm_tools.invoke(messages)

In [38]:
messages.append(ai_messages)

In [42]:
import json

for tool_call in ai_messages.tool_calls:
    if tool_call["name"] == "convert_currency":
        tool_message1 = convert_currency.invoke(tool_call)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)
    if tool_call["name"] == "convert":
        tool_call["args"]["conversion_rate"] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [44]:
messages

[HumanMessage(content='What is the conversion rate of USD to INR? Can you convert 100 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-13T03:53:30.291835Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10577884500, 'load_duration': 2870120875, 'prompt_eval_count': 277, 'prompt_eval_duration': 1335818000, 'eval_count': 96, 'eval_duration': 6356491000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a13364-b24c-7983-93fe-4aac6fb6e51d-0', tool_calls=[{'name': 'convert_currency', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '939f1778-fe7b-4cac-a4a3-b5771fd6dec6', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 100, 'conversion_rate': 95.9069}, 'id': '82a73648-07cd-4a92-9009-51ab782d120f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 277, 'outp

In [49]:
response = llm_tools.invoke(messages)
response

AIMessage(content='The conversion rate from USD to INR is approximately 95.9069. After converting 100 USD to INR, you would get approximately 9590.69 INR. Please note that the exact amount might vary slightly due to rounding.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-13T03:59:49.817933Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3538235167, 'load_duration': 8398833, 'prompt_eval_count': 904, 'prompt_eval_duration': 39413000, 'eval_count': 59, 'eval_duration': 3450325000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a1336a-985b-7a90-bf64-ac785bc5048e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 904, 'output_tokens': 59, 'total_tokens': 963})

In [55]:
response.content

'The conversion rate from USD to INR is approximately 95.9069. After converting 100 USD to INR, you would get approximately 9590.69 INR. Please note that the exact amount might vary slightly due to rounding.'